In [1]:
"""
AI Learning Navigator — V1
Rule-based / content-based next-step recommender.

Logic:
1. Load the topic graph (topics.json) — each topic has prerequisites + which
   career goals it's relevant to.
2. Given the topics a learner has completed + their goal, find all topics
   that are now "unlocked" (all prerequisites satisfied) but not yet done.
3. Rank unlocked candidates: goal-relevant first, then by graph depth
   (closer topics first), then alphabetically as a stable tiebreaker.
4. Return the top recommendation (and the full ranked list for transparency).
"""

import json
from pathlib import Path


class LearningNavigator:
    def __init__(self, topics_path: str = "topics.json"):
        data = json.loads(Path(topics_path).read_text(encoding="utf-8"))
        self.topics = data["topics"]
        self._depth_cache = {}

    def _depth(self, topic_id: str) -> int:
        """Distance from the root of the graph (topics with no prerequisites)."""
        if topic_id in self._depth_cache:
            return self._depth_cache[topic_id]
        prereqs = self.topics[topic_id]["prerequisites"]
        if not prereqs:
            depth = 0
        else:
            depth = 1 + max(self._depth(p) for p in prereqs)
        self._depth_cache[topic_id] = depth
        return depth

    def _is_unlocked(self, topic_id: str, completed: set) -> bool:
        prereqs = self.topics[topic_id]["prerequisites"]
        return all(p in completed for p in prereqs)

    def recommend(self, completed: list[str], goal: str, top_n: int = 3) -> dict:
        completed_set = set(completed)
        candidates = []

        for topic_id, info in self.topics.items():
            if topic_id in completed_set:
                continue
            if not self._is_unlocked(topic_id, completed_set):
                continue
            candidates.append(topic_id)

        def sort_key(topic_id):
            info = self.topics[topic_id]
            goal_match = 0 if goal in info["goals"] else 1  # goal-matching first
            return (goal_match, self._depth(topic_id), info["name"])

        ranked = sorted(candidates, key=sort_key)

        result = {
            "completed_count": len(completed_set),
            "goal": goal,
            "top_recommendation": None,
            "ranked_candidates": [],
        }

        for topic_id in ranked[:top_n]:
            info = self.topics[topic_id]
            result["ranked_candidates"].append(
                {
                    "id": topic_id,
                    "name": info["name"],
                    "level": info["level"],
                    "matches_goal": goal in info["goals"],
                }
            )

        if ranked:
            top = self.topics[ranked[0]]
            result["top_recommendation"] = {
                "id": ranked[0],
                "name": top["name"],
                "level": top["level"],
            }

        return result


if __name__ == "__main__":
    nav = LearningNavigator("topics.json")

    # Example matching the spec: Completed ML + first ML project -> Deep Learning
    example = nav.recommend(
        completed=["python", "math", "statistics", "data_analysis", "ml_basics", "ml_project"],
        goal="ai",
    )

    print("Input: completed ML Basics + First ML Project, goal = AI\n")
    print("Recommended Next Step:")
    print(f"  -> {example['top_recommendation']['name']} "
          f"({example['top_recommendation']['level']})\n")

    print("Other candidates:")
    for c in example["ranked_candidates"]:
        marker = "*" if c["matches_goal"] else " "
        print(f"  [{marker}] {c['name']} ({c['level']})")

Input: completed ML Basics + First ML Project, goal = AI

Recommended Next Step:
  -> Deep Learning Fundamentals (intermediate)

Other candidates:
  [*] Deep Learning Fundamentals (intermediate)
